In [1]:
query = """How does approximate nearest neighbor search work?"""

In [4]:
from embedder import Embedder
embed = Embedder()

In [6]:
v = embed.encode(query)
v[0]

np.float64(-0.020582036807885073)

### Loading the data

We pull the lesson pages from the course repository, the same way as in homework 1. We pin to commit 8c1834d so everyone works with the same data.



In [7]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

Each document is a dictionary with a `filename` and `content`, and there are 72 pages.

In [9]:
documents[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

In [10]:
documents[0]['content']

'# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simple language 

### Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed its `content`, and compute the cosine similarity with the query vector from Q1. What do you get?

In [13]:
len(documents)

72

In [17]:
for i in range(len(documents)):
    if documents[i]['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md':
        idx = i

idx 

22

In [19]:
page_v = embed.encode(documents[idx]['content'])

In [20]:
page_v.dot(v)

np.float64(0.361070280302606)

### Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

In [21]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

We embed every chunk's `content` with `encode_batch`, stack the vectors into a matrix X, and score the Q1 query against all chunks:



In [29]:
len(chunks)

295

In [24]:
contents = [chunk["content"] for chunk in chunks]

vectors = embed.encode_batch(contents)

import numpy as np
X = np.vstack(vectors)

In [25]:
scores = X.dot(v)

In [30]:
chunks[np.argmax(scores)]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

### Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use `VectorSearch` from minsearch and run a search for the following query:

> What metric do we use to evaluate a search engine?

Which file is the `filename` of the first result?

In [55]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

In [56]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

In [57]:
results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

### Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with `Index` from minsearch. Use `content` as a text field.

Run both searches for this query:

> How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

In [58]:
from minsearch import Index

index = Index(
    text_fields=["content"]
)

index.fit(chunks)

In [59]:
query = "How do I store vectors in PostgreSQL?"

search_results = index.search(
    query,
    num_results=5
)

for result in search_results:
    print(result['filename'])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [60]:
query = "How do I store vectors in PostgreSQL?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

for result in results:
    print(result['filename'])

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


### Q6. Hybrid search

Both vector and text search have their strengths and weaknesses. Vector search matches by meaning, so it finds relevant pages even when they use words different from the query. But it can miss exact terms like names, codes, or rare keywords. Text search is the opposite: it nails exact words but misses paraphrases and synonyms.

We don't have to pick one or the other - we can use both and merge their results. This approach is called "hybrid search".

Each search produces its own ranked list, so we need a way to combine them into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores the raw scores from each method, which live on different scales and aren't directly comparable. Instead, it looks only at the position of each document in each list.

Every document scores by its position (`rank`, starting at 0) in each list, and we sum the scores across lists with a constant `k = 60`:

```
RRF(d) = sum over lists of  1 / (k + rank(d))
```

"Sum over lists" means we go through every ranked list and, for each list where the document appears, add its `1 / (k + rank)` contribution. A document found by both searches collects a score from each list, while one found by only a single search collects just one.

The constant `k` controls how much the exact rank matters. A larger `k` flattens the gap between positions, so the difference between rank 0 and rank 5 counts for less. A smaller `k` does the opposite: it sharpens that gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default. You rarely need to tune it. Lower it when only the top results matter. Raise it to reward documents that appear across many lists, even when they never quite reach the top.

A document that ranks well in both lists ends up higher than one that's only strong in a single list.

In [61]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

Now run the query "How do I give the model access to tools?" with vector and text search and fuse the results with rrf:

In [66]:
query = "How do I give the model access to tools?"

In [67]:
# text search

from minsearch import Index

index = Index(
    text_fields=["content"]
)

index.fit(chunks)


text_results = index.search(
    query,
    num_results=5
)


In [68]:
# vector search
query_vector = embed.encode(query)

vec_results = vindex.search(query_vector, num_results=5)


In [69]:
rrf([text_results, vec_results], num_results=5)[0]['filename']

'01-agentic-rag/lessons/13-function-calling.md'